# UniversalCEFR: proposed fixes as code

Companion to *UniversalCEFR public collection: observations and proposed fixes* (A. Serrano, 30 Sep 2026).
This notebook runs `universalcefr_fixes.py` on the 24 public parquet exports and the public dataset cards.
It reads the published data and redistributes nothing. MIT licence.

Set `DATA_DIR` to a folder holding the exports; the first cell fetches them if they are missing (about 26 MB).

In [1]:
import os, collections
import universalcefr_fixes as F
DATA_DIR = os.environ.get("UCEFR_DIR", "../data/pq")
F.download(DATA_DIR)   # no-op for files already present
rows, cards = F.load(DATA_DIR)
print(sum(len(v) for v in rows.values()), "rows in", len(rows), "corpora")

65037 rows in 24 corpora


## (a) The 17 label strings and the six-band mapping

In [2]:
seen = collections.Counter(r["cefr_level"] for rs in rows.values() for r in rs)
for s, n in sorted(seen.items(), key=lambda x: (F.normalise_label(x[0]).kind, x[0])):
    l = F.normalise_label(s)
    print(f"{s!r:15} {n:6,}  level={l.cefr_level}  sublevel={l.cefr_sublevel}  range={l.cefr_range}  missing_reason={l.label_missing_reason}")
print(len(seen), "distinct strings")

'A1'            10,774  level=A1  sublevel=None  range=None  missing_reason=None
'A2'            14,560  level=A2  sublevel=None  range=None  missing_reason=None
'B1'            15,735  level=B1  sublevel=None  range=None  missing_reason=None
'B2'            13,252  level=B2  sublevel=None  range=None  missing_reason=None
'C1'             7,515  level=C1  sublevel=None  range=None  missing_reason=None
'C2'             1,338  level=C2  sublevel=None  range=None  missing_reason=None
'B'                 18  level=None  sublevel=None  range=B1-B2  missing_reason=None
'EMPTY'              3  level=None  sublevel=None  range=None  missing_reason=empty
'NA'                 7  level=None  sublevel=None  range=None  missing_reason=na
'Unassessable'       6  level=None  sublevel=None  range=None  missing_reason=unassessable
'unrated'            6  level=None  sublevel=None  range=None  missing_reason=unrated
'A1+'               61  level=A1  sublevel=A1+  range=None  missing_reason=None
'A2+'   

Applying the fix to one row keeps the original string alongside the new fields:

In [3]:
F.apply_label_fix({"cefr_level": "B1+", "license": "CC BY-NC-SA 4.0", "text": "..."})

{'cefr_level': 'B1',
 'license': 'CC BY-NC-SA 4.0',
 'text': '...',
 'cefr_sublevel': 'B1+',
 'cefr_range': None,
 'label_missing_reason': None,
 'cefr_level_original': 'B1+'}

## (b) Rows outside what the collection and the cards declare

In [4]:
F.print_flags(rows, cards)


check_row(): rows outside what the collection and the cards declare
  label-outside-cefr           40 rows in 5 corpora
  licence-substantive       5,527 rows in 5 corpora
  licence-unparseable       1,372 rows in 1 corpora
  licence-word_order       11,108 rows in 7 corpora
  rows with at least one flag: 18,034 of 65,037


## (c) Rows affected by each proposed change, per corpus

In [5]:
import pandas as pd
rep = pd.DataFrame(F.report(rows, cards)).set_index("corpus")
cols = [k for k, _ in F.CHANGES]
rep[["rows", "distinct_texts"] + cols].loc[:, lambda d: (d != 0).any()]

,rows,distinct_texts,label_plus_to_sublevel,label_band_only_to_range,label_missing_to_null,licence_word_order,licence_substantive,licence_unparseable,licence_to_spdx,collapse_band_expanded,collapse_rater_rows,placeholder_none,cross_corpus_exact
corpus,,,,,,,,,,,,,
caes_es,31149,20629,0,0,0,0,0,0,31149,0,0,214,15
hablacultura_es,713,667,0,18,0,0,0,0,713,0,0,0,0
kwiqiz_es,206,206,0,0,0,0,0,0,206,0,0,0,0
cople2_pt,942,471,0,0,0,942,0,0,942,471,0,0,0
peapl2_pt,481,481,0,0,0,481,0,0,481,0,0,0,0
readme_en,2822,2820,0,0,0,2822,0,0,2822,0,0,0,1
readme_fr,1669,1668,0,0,0,1669,0,0,1669,0,0,0,0
readme_ru,1758,1751,0,0,0,1758,0,0,1758,0,0,0,0
readme_ar,1945,1939,0,0,0,1945,0,0,1945,0,0,0,0


In [6]:
rep[cols].sum().rename("rows affected").to_frame()

,rows affected
label_plus_to_sublevel,1823
label_band_only_to_range,18
label_missing_to_null,22
licence_word_order,11108
licence_substantive,5527
licence_unparseable,1372
licence_to_spdx,65037
collapse_band_expanded,471
collapse_rater_rows,2401
placeholder_none,214
